<a href="https://colab.research.google.com/github/Digital-AI-Finance/Cryptocurrency/blob/main/notebooks/deploy_token.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Deploy Your Own Crypto Token

This notebook walks you through deploying a minimal ERC-20 token on the Sepolia testnet using Python.
You will compile Solidity, sign a deploy transaction, and interact with your token on-chain.

## What You Need

- **MetaMask wallet** with a Sepolia account (export the private key under Account Details)
- **Sepolia test ETH** for gas fees (free from [Google Cloud faucet](https://cloud.google.com/application/web3/faucet/ethereum/sepolia) or [Alchemy faucet](https://sepoliafaucet.com))
- *(Optional)* A free [Infura](https://infura.io) or [Alchemy](https://alchemy.com) API key for a more reliable RPC endpoint

> **Security warning:** Never use a wallet that holds real funds. Create a fresh throwaway wallet for testnet work.

In [ ]:
!pip install -q web3 py-solc-x

import solcx
solcx.install_solc("0.8.20")
print("✓ Solidity compiler 0.8.20 installed")

In [ ]:
from web3 import Web3
import solcx

# Connect to Sepolia testnet
# Replace with your own Infura/Alchemy URL for better reliability:
# PROVIDER_URL = "https://sepolia.infura.io/v3/YOUR_PROJECT_ID"
PROVIDER_URL = "https://rpc.sepolia.org"

w3 = Web3(Web3.HTTPProvider(PROVIDER_URL))

print("=" * 60)
print("SEPOLIA CONNECTION")
print("=" * 60)
if w3.is_connected():
    print(f"✓ Connected   | Chain ID: {w3.eth.chain_id}")
    print(f"✓ Latest block: {w3.eth.block_number:,}")
else:
    print("✗ Connection failed - try a different RPC URL")
print("=" * 60)

## The Contract

A self-contained ERC-20 token with `transfer`, `approve`, and `transferFrom`.
No OpenZeppelin imports needed — everything is defined inline so `py-solc-x` can compile it directly.

In [ ]:
CONTRACT_SOURCE = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract MyToken {
    string public name = "MyToken";
    string public symbol = "MTK";
    uint8 public decimals = 18;
    uint256 public totalSupply;
    mapping(address => uint256) public balanceOf;
    mapping(address => mapping(address => uint256)) public allowance;

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);

    constructor(uint256 initialSupply) {
        totalSupply = initialSupply * 10 ** decimals;
        balanceOf[msg.sender] = totalSupply;
        emit Transfer(address(0), msg.sender, totalSupply);
    }

    function transfer(address to, uint256 amount) public returns (bool) {
        require(balanceOf[msg.sender] >= amount, "Insufficient balance");
        balanceOf[msg.sender] -= amount;
        balanceOf[to] += amount;
        emit Transfer(msg.sender, to, amount);
        return true;
    }

    function approve(address spender, uint256 amount) public returns (bool) {
        allowance[msg.sender][spender] = amount;
        emit Approval(msg.sender, spender, amount);
        return true;
    }

    function transferFrom(address from, address to, uint256 amount) public returns (bool) {
        require(balanceOf[from] >= amount, "Insufficient balance");
        require(allowance[from][msg.sender] >= amount, "Insufficient allowance");
        balanceOf[from] -= amount;
        balanceOf[to] += amount;
        allowance[from][msg.sender] -= amount;
        emit Transfer(from, to, amount);
        return true;
    }
}
"""

compiled = solcx.compile_source(
    CONTRACT_SOURCE,
    solc_version="0.8.20",
    output_values=["abi", "bin"],
)
contract_interface = compiled["<stdin>:MyToken"]
ABI = contract_interface["abi"]
BYTECODE = contract_interface["bin"]

print("=" * 60)
print("COMPILATION")
print("=" * 60)
print(f"✓ Contract compiled successfully")
print(f"✓ ABI entries : {len(ABI)}")
print(f"✓ Bytecode    : {len(BYTECODE) // 2} bytes")
print("=" * 60)

## Deploy

Paste your **Sepolia** private key when prompted. The key is read with `getpass` and never displayed.

> **Never use a wallet with real funds.** Create a dedicated testnet wallet.

In [ ]:
import getpass

PRIVATE_KEY = getpass.getpass("Paste your Sepolia private key: ")
account = w3.eth.account.from_key(PRIVATE_KEY)
DEPLOYER = account.address

print(f"✓ Deployer address: {DEPLOYER}")
print(f"✓ Balance: {w3.from_wei(w3.eth.get_balance(DEPLOYER), 'ether'):.4f} ETH")

# Build deploy transaction
INITIAL_SUPPLY = 500_000  # 500 000 tokens

Token = w3.eth.contract(abi=ABI, bytecode=BYTECODE)
tx = Token.constructor(INITIAL_SUPPLY).build_transaction({
    "from": DEPLOYER,
    "nonce": w3.eth.get_transaction_count(DEPLOYER),
    "chainId": 11155111,
})

# Sign and send
signed = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
print(f"\nTransaction sent: {tx_hash.hex()}")
print("Waiting for confirmation...")

receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
CONTRACT_ADDRESS = receipt.contractAddress

print()
print("=" * 60)
print("DEPLOYMENT SUCCESSFUL")
print("=" * 60)
print(f"✓ Contract address : {CONTRACT_ADDRESS}")
print(f"✓ Block number     : {receipt.blockNumber:,}")
print(f"✓ Gas used         : {receipt.gasUsed:,}")
print(f"✓ Etherscan        : https://sepolia.etherscan.io/address/{CONTRACT_ADDRESS}")
print("=" * 60)

In [ ]:
token = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)

name       = token.functions.name().call()
symbol     = token.functions.symbol().call()
decimals   = token.functions.decimals().call()
supply_raw = token.functions.totalSupply().call()
balance_raw = token.functions.balanceOf(DEPLOYER).call()

supply  = supply_raw / 10**decimals
balance = balance_raw / 10**decimals

print("=" * 60)
print("YOUR TOKEN ON-CHAIN")
print("=" * 60)
print(f"✓ Name         : {name}")
print(f"✓ Symbol       : {symbol}")
print(f"✓ Decimals     : {decimals}")
print(f"✓ Total supply : {supply:,.0f} {symbol}")
print(f"✓ Your balance : {balance:,.0f} {symbol}")
print("=" * 60)

## What's Next?

- **View on Etherscan** — paste the contract address at [sepolia.etherscan.io](https://sepolia.etherscan.io)
- **Add to MetaMask** — Import Token > paste the contract address to see your balance in the wallet
- **Try a transfer** — call `token.functions.transfer(recipient, amount)` to send tokens to another address
- **Verify the contract** — flatten the source and verify on Etherscan so others can read the code